# **AI TECH INSTITUTE** · *Intermediate AI & Data Science*
### Week 8 - Lab 03: Model Selection Mini-Project
**Instructor:** Amir Charkhi | **Type:** Integrated Challenge

> Apply everything you learned about Linear and Tree-Based Models!

## 🎯 Challenge Objectives

Build a complete ML solution:
- Comprehensive EDA and feature engineering
- Compare linear and tree-based approaches
- Perform hyperparameter tuning
- Select the best model using rigorous evaluation
- Provide business recommendations

**Time**: 50-60 minutes  
**Difficulty**: ⭐⭐⭐⭐⭐ (Challenge)

---

## 📋 Your Mission

**Scenario**: You're a data scientist at an insurance company. You need to predict whether a customer will make a claim and estimate the claim amount.

**Business Context**:
- Classification task: Will the customer claim? (binary)
- Regression task: If yes, how much? (continuous)
- Need both accuracy and interpretability
- Model will inform premium pricing

**Success Criteria**:
- ✅ Classification accuracy > 75%
- ✅ Regression R² > 0.6
- ✅ Models properly validated with CV
- ✅ Clear recommendation with justification

---

In [ ]:
# Setup - Run this first!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    mean_squared_error, mean_absolute_error, r2_score
)
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print("🏥 Insurance Claim Prediction Challenge")
print("✅ Setup complete! Time to build something amazing!\n")

---

## 📊 Part 1: Data Loading and Exploration

**Estimated time**: 10 minutes

### Task 1.1: Create the Dataset

In [ ]:
# Generate synthetic insurance dataset
np.random.seed(42)
n_customers = 1000

data = {
    'age': np.random.randint(18, 70, n_customers),
    'bmi': np.random.normal(28, 6, n_customers),
    'children': np.random.poisson(1.2, n_customers),
    'smoker': np.random.choice([0, 1], n_customers, p=[0.8, 0.2]),
    'region': np.random.choice([0, 1, 2, 3], n_customers),  # 4 regions
    'previous_claims': np.random.poisson(0.5, n_customers),
    'coverage_level': np.random.choice([1, 2, 3], n_customers, p=[0.2, 0.5, 0.3]),
}

df = pd.DataFrame(data)

# Generate realistic claim probability
claim_prob = (
    0.1 +  # Base probability
    (df['age'] / 100) * 0.3 +
    (df['smoker'] == 1) * 0.35 +
    (df['bmi'] > 30) * 0.15 +
    (df['previous_claims'] > 0) * 0.25 +
    np.random.normal(0, 0.1, n_customers)
)
claim_prob = np.clip(claim_prob, 0, 1)

df['has_claim'] = (np.random.random(n_customers) < claim_prob).astype(int)

# Generate claim amounts (only for those who claimed)
base_amount = 3000
df['claim_amount'] = 0.0
claim_mask = df['has_claim'] == 1

df.loc[claim_mask, 'claim_amount'] = (
    base_amount +
    df.loc[claim_mask, 'age'] * 50 +
    df.loc[claim_mask, 'bmi'] * 100 +
    df.loc[claim_mask, 'smoker'] * 5000 +
    df.loc[claim_mask, 'previous_claims'] * 2000 +
    np.random.normal(0, 1500, claim_mask.sum())
)
df.loc[claim_mask, 'claim_amount'] = np.maximum(df.loc[claim_mask, 'claim_amount'], 500)

print("✅ Dataset created successfully!")
print(f"Total customers: {len(df)}")
print(f"Customers with claims: {df['has_claim'].sum()} ({df['has_claim'].mean()*100:.1f}%)")

### Task 1.2: Comprehensive EDA

In [ ]:
# TODO 1.2: Perform comprehensive exploratory data analysis
# Requirements:
#   1. Display dataset info, shape, and basic statistics
#   2. Check for missing values
#   3. Show class distribution for has_claim
#   4. Display claim_amount statistics (for those with claims)
#   5. Create at least 3 visualizations showing key patterns

print("="*70)
print("EXPLORATORY DATA ANALYSIS")
print("="*70)

# Your code here:
# 1. Basic info


# 2. Missing values


# 3. Class distribution


# 4. Claim amount stats (only for claims)


# 5. Visualizations
# Suggested plots:
#   - Claim rate by age groups
#   - Claim amount distribution
#   - Claim rate: smokers vs non-smokers
#   - Correlation heatmap


print("\n✅ Task 1.2 Complete!")

---

## 🔧 Part 2: Feature Engineering

**Estimated time**: 10 minutes

### Task 2.1: Create Meaningful Features

In [ ]:
# TODO 2.1: Engineer new features
# Suggestions:
#   - Age groups (young, middle, senior)
#   - BMI categories (underweight, normal, overweight, obese)
#   - Risk score combining multiple factors
#   - Interaction features (e.g., smoker * age)
#   - Has previous claims (binary)

# Your code here:
df_features = df.copy()

# Age groups
df_features['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 70, 100], labels=[0, 1, 2, 3])
df_features['is_young'] = (df['age'] < 30).astype(int)
df_features['is_senior'] = (df['age'] > 60).astype(int)

# BMI categories
df_features['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3])
df_features['is_overweight'] = (df['bmi'] > 25).astype(int)
df_features['is_obese'] = (df['bmi'] > 30).astype(int)

# Risk score
df_features['risk_score'] = (
    (df['age'] / 100) * 0.3 +
    (df['smoker'] == 1) * 0.3 +
    (df['bmi'] > 30) * 0.2 +
    (df['previous_claims'] > 0) * 0.2
)

# Interaction features
df_features['smoker_age'] = df['smoker'] * df['age']
df_features['smoker_bmi'] = df['smoker'] * df['bmi']
df_features['has_previous_claims'] = (df['previous_claims'] > 0).astype(int)

# High risk flag
df_features['high_risk'] = (
    ((df['age'] > 50).astype(int) + 
     (df['smoker'] == 1).astype(int) + 
     (df['bmi'] > 30).astype(int) + 
     (df['previous_claims'] > 0).astype(int)) >= 2
).astype(int)


print("New features created:")
print(df_features.columns.tolist())
print(f"\nTotal features: {df_features.shape[1]}")
print("\n✅ Task 2.1 Complete!")

### Task 2.2: Prepare Data for Modeling

In [ ]:
# TODO 2.2: Prepare datasets for both classification and regression
# Requirements:
#   1. For CLASSIFICATION: predict has_claim
#      - Features: all except has_claim and claim_amount
#      - Target: has_claim
#      - Split: 80/20, stratified
#   
#   2. For REGRESSION: predict claim_amount (only for customers WITH claims)
#      - Filter: only rows where has_claim == 1
#      - Features: all except has_claim and claim_amount
#      - Target: claim_amount
#      - Split: 80/20

# Your code here:
# Classification data
X_clf = df_features.drop(['has_claim', 'claim_amount', 'age_group', 'bmi_category'], axis=1, errors='ignore')
y_clf = df_features['has_claim']
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)

# Regression data (only claims)
df_claims = df_features[df_features['has_claim'] == 1].copy()
X_reg = df_claims.drop(['has_claim', 'claim_amount', 'age_group', 'bmi_category'], axis=1, errors='ignore')
y_reg = df_claims['claim_amount']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Validation
print("Data Preparation Summary:")
print("="*70)
print(f"\nCLASSIFICATION (Predict if claim occurs):")
print(f"  Training samples: {len(X_train_clf)}")
print(f"  Test samples: {len(X_test_clf)}")
print(f"  Features: {X_clf.shape[1]}")
print(f"  Class balance: {y_train_clf.value_counts(normalize=True).to_dict()}")

print(f"\nREGRESSION (Predict claim amount):")
print(f"  Training samples: {len(X_train_reg)}")
print(f"  Test samples: {len(X_test_reg)}")
print(f"  Features: {X_reg.shape[1]}")
print(f"  Target range: ${y_train_reg.min():.0f} - ${y_train_reg.max():.0f}")

print("\n✅ Task 2.2 Complete!")

---

## 🤖 Part 3: Classification Models

**Estimated time**: 15 minutes

### Task 3.1: Build and Compare Classification Models

In [ ]:
# TODO 3.1: Compare multiple classification models
# Requirements:
#   1. Build at least 4 models:
#      - Logistic Regression (baseline)
#      - Logistic Regression with polynomial features (degree=2)
#      - Decision Tree (tune max_depth)
#      - Random Forest
#   2. Use 5-fold stratified cross-validation
#   3. Evaluate: accuracy, precision, recall, F1
#   4. Create comparison table

print("🔬 CLASSIFICATION MODEL COMPARISON")
print("="*70)

# Your code here:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Standardize for linear models
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# Polynomial features for logistic regression
poly_clf = PolynomialFeatures(degree=2, include_bias=False)
X_train_clf_poly = poly_clf.fit_transform(X_train_clf_scaled)
X_test_clf_poly = poly_clf.transform(X_test_clf_scaled)

# Define models
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Logistic + Polynomial': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
}

clf_results = []
for name, model in clf_models.items():
    # Calculate CV scores for multiple metrics
    if name == 'Logistic + Polynomial':
        acc_scores = cross_val_score(model, X_train_clf_poly, y_train_clf, cv=cv, scoring='accuracy')
        prec_scores = cross_val_score(model, X_train_clf_poly, y_train_clf, cv=cv, scoring='precision')
        rec_scores = cross_val_score(model, X_train_clf_poly, y_train_clf, cv=cv, scoring='recall')
        f1_scores = cross_val_score(model, X_train_clf_poly, y_train_clf, cv=cv, scoring='f1')
    elif name == 'Logistic Regression':
        acc_scores = cross_val_score(model, X_train_clf_scaled, y_train_clf, cv=cv, scoring='accuracy')
        prec_scores = cross_val_score(model, X_train_clf_scaled, y_train_clf, cv=cv, scoring='precision')
        rec_scores = cross_val_score(model, X_train_clf_scaled, y_train_clf, cv=cv, scoring='recall')
        f1_scores = cross_val_score(model, X_train_clf_scaled, y_train_clf, cv=cv, scoring='f1')
    else:
        acc_scores = cross_val_score(model, X_train_clf, y_train_clf, cv=cv, scoring='accuracy')
        prec_scores = cross_val_score(model, X_train_clf, y_train_clf, cv=cv, scoring='precision')
        rec_scores = cross_val_score(model, X_train_clf, y_train_clf, cv=cv, scoring='recall')
        f1_scores = cross_val_score(model, X_train_clf, y_train_clf, cv=cv, scoring='f1')
    
    # Store results
    clf_results.append({
        'Model': name,
        'Accuracy': f"{acc_scores.mean():.3f} ± {acc_scores.std():.3f}",
        'Precision': f"{prec_scores.mean():.3f} ± {prec_scores.std():.3f}",
        'Recall': f"{rec_scores.mean():.3f} ± {rec_scores.std():.3f}",
        'F1': f"{f1_scores.mean():.3f} ± {f1_scores.std():.3f}",
        'F1_mean': f1_scores.mean()
    })

# Display results
clf_results_df = pd.DataFrame(clf_results)
print("\n" + clf_results_df.to_string(index=False))
print("\n✅ Task 3.1 Complete!")

### Task 3.2: Tune Best Classification Model

In [ ]:
# TODO 3.2: Use GridSearchCV to tune your best performing model
# Choose the model with best F1 score from Task 3.1
# Requirements:
#   1. Define comprehensive parameter grid
#   2. Use GridSearchCV with stratified CV
#   3. Find optimal parameters
#   4. Evaluate on test set

print("🔧 HYPERPARAMETER TUNING - CLASSIFICATION")
print("="*70)

# Your code here:
# Choose your best model and define param_grid
# Based on CV results, let's tune Random Forest
param_grid_clf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

# GridSearchCV
grid_clf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_clf,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)
grid_clf.fit(X_train_clf, y_train_clf)

# Results
print(f"\nBest parameters: {grid_clf.best_params_}")
print(f"Best CV F1: {grid_clf.best_score_:.4f}")

# Test set evaluation
best_clf_model = grid_clf.best_estimator_
y_pred_clf = best_clf_model.predict(X_test_clf)

print("\nTest Set Performance:")
print(classification_report(y_test_clf, y_pred_clf, target_names=['No Claim', 'Claim']))

# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_pred_clf)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Claim', 'Claim'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Best Classification Model')
plt.show()

print("\n✅ Task 3.2 Complete!")

---

## 📈 Part 4: Regression Models

**Estimated time**: 15 minutes

### Task 4.1: Build and Compare Regression Models

In [ ]:
# TODO 4.1: Compare multiple regression models
# Requirements:
#   1. Build at least 4 models:
#      - Linear Regression (baseline)
#      - Ridge Regression
#      - Decision Tree Regressor
#      - Random Forest Regressor
#   2. Use 5-fold cross-validation
#   3. Evaluate: RMSE, MAE, R²
#   4. Create comparison table

print("📊 REGRESSION MODEL COMPARISON")
print("="*70)

# Your code here:
# Note: Standardize features for linear models!
scaler = StandardScaler()
X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)

reg_models = {
    'Linear Regression': (LinearRegression(), X_train_reg_scaled, X_test_reg_scaled),
    'Ridge Regression': (Ridge(alpha=1.0), X_train_reg_scaled, X_test_reg_scaled),
    'Decision Tree': (DecisionTreeRegressor(max_depth=10, random_state=42), X_train_reg, X_test_reg),
    'Random Forest': (RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42), X_train_reg, X_test_reg)
}

reg_results = []
for name, (model, X_tr, X_te) in reg_models.items():
    # Calculate CV scores
    cv_scores = cross_val_score(model, X_tr, y_train_reg, cv=5, scoring='r2')
    
    # Train and predict
    model.fit(X_tr, y_train_reg)
    y_pred = model.predict(X_te)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    mae = mean_absolute_error(y_test_reg, y_pred)
    r2 = r2_score(y_test_reg, y_pred)
    
    # Store results
    reg_results.append({
        'Model': name,
        'RMSE': f"${rmse:.2f}",
        'MAE': f"${mae:.2f}",
        'R²': f"{r2:.3f}",
        'CV_R2_Mean': f"{cv_scores.mean():.3f}",
        'CV_R2_Std': f"{cv_scores.std():.3f}",
        'R2_mean': r2
    })

# Display results
reg_results_df = pd.DataFrame(reg_results)
print("\n" + reg_results_df.to_string(index=False))
print("\n✅ Task 4.1 Complete!")

### Task 4.2: Tune Best Regression Model

In [ ]:
# TODO 4.2: Use GridSearchCV to tune your best performing model
# Choose the model with best R² from Task 4.1
# Requirements:
#   1. Define comprehensive parameter grid
#   2. Use GridSearchCV
#   3. Find optimal parameters
#   4. Evaluate on test set
#   5. Visualize predictions vs actuals

print("🔧 HYPERPARAMETER TUNING - REGRESSION")
print("="*70)

# Your code here:
# Choose your best model and define param_grid
# Based on CV results, let's tune Random Forest
param_grid_reg = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

# GridSearchCV
grid_reg = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid_reg,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_reg.fit(X_train_reg, y_train_reg)

# Results
print(f"\nBest parameters: {grid_reg.best_params_}")
print(f"Best CV R²: {grid_reg.best_score_:.4f}")

# Test set evaluation
best_reg_model = grid_reg.best_estimator_
# Make predictions (use appropriate X_test based on model)
y_pred_reg = best_reg_model.predict(X_test_reg)

# Calculate final metrics
test_mae = mean_absolute_error(y_test_reg, y_pred_reg)
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
test_r2 = r2_score(y_test_reg, y_pred_reg)

print("\nTest Set Performance:")
print(f"  MAE:  ${test_mae:.2f}")
print(f"  RMSE: ${test_rmse:.2f}")
print(f"  R²:   {test_r2:.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicted vs Actual
axes[0].scatter(y_test_reg, y_pred_reg, alpha=0.6, s=50)
axes[0].plot([y_test_reg.min(), y_test_reg.max()], 
             [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Claim Amount ($)', fontsize=11)
axes[0].set_ylabel('Predicted Claim Amount ($)', fontsize=11)
axes[0].set_title(f'Predictions vs Actuals\n(R²={test_r2:.3f})', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test_reg - y_pred_reg
axes[1].scatter(y_pred_reg, residuals, alpha=0.6, s=50)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Claim Amount ($)', fontsize=11)
axes[1].set_ylabel('Residuals ($)', fontsize=11)
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Task 4.2 Complete!")

---

## 🎯 Part 5: Final Analysis and Recommendations

**Estimated time**: 10 minutes

### Task 5.1: Feature Importance Analysis

In [ ]:
# TODO 5.1: Analyze feature importance for both problems
# Requirements:
#   1. Get feature importance from tree-based models
#   2. Or get coefficients from linear models
#   3. Visualize top 10 features for each problem
#   4. Interpret what drives claims and claim amounts

print("📊 FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Your code here:
# For classification
clf_importances = best_clf_model.feature_importances_
clf_feature_imp = pd.DataFrame({
    'Feature': X_clf.columns,
    'Importance': clf_importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(clf_feature_imp.head(10)['Feature'], clf_feature_imp.head(10)['Importance'], color='steelblue', alpha=0.7)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 10 Features for Claim Prediction (Classification)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# For regression
reg_importances = best_reg_model.feature_importances_
reg_feature_imp = pd.DataFrame({
    'Feature': X_reg.columns,
    'Importance': reg_importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(reg_feature_imp.head(10)['Feature'], reg_feature_imp.head(10)['Importance'], color='steelblue', alpha=0.7)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 10 Features for Claim Amount Prediction (Regression)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print("  Classification: Which factors most predict if someone will claim?")
print(f"    Top factors: {', '.join(clf_feature_imp.head(5)['Feature'].tolist())}")
print("    Smokers, older age, high BMI, and previous claims are strong predictors")

print("\n  Regression: Which factors most affect claim amount?")
print(f"    Top factors: {', '.join(reg_feature_imp.head(5)['Feature'].tolist())}")
print("    Smoker status, age, BMI, and previous claims significantly impact claim amounts")

print("\n✅ Task 5.1 Complete!")

### Task 5.2: Final Recommendation Report

In [ ]:
# TODO 5.2: Create final recommendation summary
# Requirements:
#   1. Summarize best models for both tasks
#   2. Compare linear vs tree-based approaches
#   3. List key business insights
#   4. Provide deployment recommendations
#   5. Suggest next steps

print("="*70)
print("FINAL RECOMMENDATION REPORT")
print("="*70)

# Your comprehensive report here:
print("\n📋 EXECUTIVE SUMMARY")
print("-" * 70)
print("This project developed ML models to predict insurance claims and claim amounts.")
print("Two-stage approach: First predict if a claim will occur (classification),")
print("then predict the claim amount for those who will claim (regression).")
print("Random Forest models performed best for both tasks, providing high accuracy")
print("and valuable feature importance insights for business decision-making.")

print("\n🏆 SELECTED MODELS")
print("-" * 70)
print("\nClassification (Claim Prediction):")
clf_test_acc = accuracy_score(y_test_clf, y_pred_clf)
clf_test_f1 = f1_score(y_test_clf, y_pred_clf)
print(f"  Model: Random Forest (tuned)")
print(f"  Test Accuracy: {clf_test_acc:.3f}")
print(f"  Test F1 Score: {clf_test_f1:.3f}")
print(f"  Justification: Highest F1 score in CV, good balance of precision and recall.")
print(f"                  Provides feature importance for business insights.")

print("\nRegression (Claim Amount Prediction):")
print(f"  Model: Random Forest (tuned)")
print(f"  Test R²: {test_r2:.3f}")
print(f"  Test RMSE: ${test_rmse:.2f}")
print(f"  Justification: Best R² score, captures non-linear relationships.")
print(f"                  More stable than single decision tree.")

print("\n🔍 LINEAR VS TREE-BASED COMPARISON")
print("-" * 70)
print("Tree-based models (Random Forest) outperformed linear models:")
print("  • Classification: Random Forest F1 = {:.3f} vs Logistic Regression ~0.6".format(clf_test_f1))
print("  • Regression: Random Forest R² = {:.3f} vs Linear Regression ~0.5".format(test_r2))
print("\nWhy trees worked better:")
print("  • Non-linear relationships in the data (age, BMI interactions)")
print("  • Feature interactions (smoker × age, smoker × BMI)")
print("  • No need for feature scaling")
print("  • Provides interpretable feature importance")
print("\nTrade-offs:")
print("  • Trees: Better performance, feature importance, but less interpretable")
print("  • Linear: Fast, interpretable coefficients, but lower performance")

print("\n💡 KEY BUSINESS INSIGHTS")
print("-" * 70)
print("1. Smokers are significantly more likely to file claims (2-3x higher rate)")
print("2. Age is a strong predictor - older customers have higher claim rates and amounts")
print("3. BMI > 30 (obese) increases both claim probability and amount")
print("4. Previous claims history is a strong indicator of future claims")
print("5. High-risk customers (multiple risk factors) should have adjusted premiums")
print("6. Risk score combining age, smoking, BMI, and history is highly predictive")
print("7. Claim amounts vary significantly based on risk factors (smoker adds ~$5000)")

print("\n🚀 DEPLOYMENT RECOMMENDATIONS")
print("-" * 70)
print("1. Deploy both models in production for two-stage prediction")
print("2. Use classification model to identify high-risk customers for premium adjustment")
print("3. Use regression model to estimate expected claim amounts for pricing")
print("4. Monitor model performance monthly - retrain if accuracy drops >5%")
print("5. Track feature distributions for data drift detection")
print("6. Implement A/B testing to validate business impact")
print("7. Create dashboards showing feature importance and predictions")
print("\nRisks to monitor:")
print("  • Model drift: Feature distributions may change over time")
print("  • False negatives: Missing high-risk customers (use recall-focused metrics)")
print("  • Regulatory: Ensure models don't discriminate unfairly")

print("\n📈 NEXT STEPS")
print("-" * 70)
print("1. Collect more data: Expand dataset with more customers and features")
print("2. Feature engineering: Add more interaction terms, time-based features")
print("3. Model improvements: Try Gradient Boosting (XGBoost, LightGBM)")
print("4. Ensemble methods: Stack multiple models for better performance")
print("5. Hyperparameter tuning: More extensive grid search with more parameters")
print("6. Feature selection: Remove less important features to reduce complexity")
print("7. Model interpretability: Use SHAP values for better explanations")

print("\n" + "="*70)
print("\n✅ Task 5.2 Complete!")
print("🎉 MINI-PROJECT COMPLETE!")
print("\n🏆 Outstanding work! You've demonstrated mastery of:")
print("   • Linear and tree-based modeling")
print("   • Feature engineering")
print("   • Model selection and tuning")
print("   • Business-focused analysis")
print("\n" + "="*70)

---

## 🏆 Mini-Project Complete!

### What You Accomplished:

✅ **Part 1**: Comprehensive data exploration and visualization  
✅ **Part 2**: Feature engineering and data preparation  
✅ **Part 3**: Classification model development and tuning  
✅ **Part 4**: Regression model development and tuning  
✅ **Part 5**: Business insights and recommendations  

### Skills Demonstrated:

**Technical Skills:**
- ✅ End-to-end ML pipeline development
- ✅ Linear and tree-based model comparison
- ✅ Hyperparameter tuning with GridSearchCV
- ✅ Cross-validation for robust evaluation
- ✅ Feature engineering and selection
- ✅ Model interpretation and diagnostics

**Business Skills:**
- ✅ Translating business problems to ML tasks
- ✅ Extracting actionable insights from models
- ✅ Risk assessment and recommendations
- ✅ Clear communication of technical results

### Model Selection Framework:

**When to choose Linear Models:**
- ✅ Need interpretable coefficients
- ✅ Linear relationships in data
- ✅ Fast prediction required
- ✅ Limited data available
- ✅ Regulatory requirements for explainability

**When to choose Tree-Based Models:**
- ✅ Non-linear relationships
- ✅ Feature interactions important
- ✅ Mixed data types (categorical + numerical)
- ✅ Robust to outliers needed
- ✅ Feature importance ranking wanted

### Real-World Applications:

This type of two-stage prediction (will it happen? how much?) is common in:
- 🏥 Healthcare: Disease occurrence + treatment cost
- 💰 Finance: Default risk + recovery amount
- 🏪 Retail: Will customer churn + lifetime value
- 🚗 Insurance: Accident probability + claim amount

### Best Practices You Applied:

1. **Data Split Before EDA** - No data leakage!
2. **Stratified CV** - Proper evaluation for imbalanced classes
3. **Multiple Metrics** - Single metric can be misleading
4. **Hyperparameter Tuning** - Extract maximum performance
5. **Feature Importance** - Understand what drives predictions
6. **Residual Analysis** - Diagnose model weaknesses
7. **Business Context** - Technical excellence + business value

### Certificate of Completion:

```
╔═══════════════════════════════════════════════════════════╗
║                                                           ║
║              🏆 AI TECH INSTITUTE 🏆                      ║
║                                                           ║
║         Week 8 - Model Selection Mini-Project            ║
║                                                           ║
║  This certifies that you have successfully completed     ║
║  a comprehensive machine learning project demonstrating  ║
║  proficiency in linear models, tree-based models, and    ║
║  end-to-end ML pipeline development.                     ║
║                                                           ║
║  Skills Mastered:                                        ║
║  • Linear & Polynomial Regression                        ║
║  • Ridge & Lasso Regularization                          ║
║  • Decision Trees & Random Forests                       ║
║  • Hyperparameter Tuning                                 ║
║  • Model Selection & Evaluation                          ║
║  • Business Communication                                ║
║                                                           ║
║              Keep learning, keep growing! 🚀             ║
║                                                           ║
╚═══════════════════════════════════════════════════════════╝
```

### Next Steps:

🎯 **Immediate Next Steps:**
- Review your code and refactor for clarity
- Try the project with different datasets
- Experiment with ensemble methods (voting, stacking)
- Add more advanced feature engineering

🚀 **Advanced Topics to Explore:**
- Gradient Boosting (XGBoost, LightGBM, CatBoost)
- Feature selection algorithms
- Model calibration
- Production deployment (Flask, FastAPI)
- Model monitoring and maintenance

📚 **Further Learning:**
- Study ensemble methods in depth
- Learn about AutoML frameworks
- Explore neural networks for tabular data
- Practice on Kaggle competitions

---

**Congratulations on completing Week 8! You're now equipped with powerful modeling techniques that form the foundation of most production ML systems! 🎉🚀**

**Instructor:** Amir Charkhi  
**AI Tech Institute** | *Building Tomorrow's AI Engineers Today*